# Snake Detective: Intro to pandas
A 25–30 minute Statistics & Data Science Club workshop.

Run cells from top to bottom using **Shift+Enter**. Predict, run, then change one thing.
All snake records are fictional; no biological or safety conclusions are intended.

The first cell recreates the original CSV. Later cells export new files into `snake_data/`.
Python basics assumed: variables, comparisons, and function calls. No pandas experience needed.

If pandas is missing, run the optional install cell once. In Colab it is usually already installed.

In [1]:
# Uncomment only if pandas is missing, then restart the kernel if prompted.
# %pip install pandas

## 1. Create our tiny CSV
Run this once to create the dataset. Each row represents one fictional individual snake; one observation is accidentally repeated. Predict how many unique snakes we have.

In [2]:
from pathlib import Path
import csv
import pandas as pd

# Fictional teaching data, NOT biological or animal-handling guidance.
# Paths are relative to the current working folder (also works in Colab).
DATA_DIR = Path("snake_data")
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / "snakes.csv"
COLUMNS = ['snake_id', 'name', 'species', 'habitat', 'length_cm', 'mass_g', 'venomous']
ROWS = [['S01', 'Noodle', 'Corn snake', 'Forest', 110, 650, False], ['S02', 'Pretzel', 'Ball python', 'Grassland', 125, 1400, False], ['S03', 'Socks', 'Garter snake', 'Wetland', 55, 140, False], ['S04', 'Ribbon', 'Kingsnake', ' Forest ', 95, 500, False], ['S05', 'Mochi', 'Ball python', 'Grassland', 100, 1100, False], ['S06', 'Zigzag', 'Viper', 'Forest', 70, 400, True], ['S07', 'Pesto', 'Tree snake', 'forest', 85, 220, False], ['S08', 'Pepper', 'Cobra', 'Grassland', 150, 1200, True], ['S09', 'Bean', 'Corn snake', 'Forest', None, 450, False], ['S10', 'Slinky', 'Water snake', 'Wetland', 80, 350, False], ['S11', 'Pickle', 'Kingsnake', 'Forest', 120, 750, False], ['S12', 'Waffles', 'Viper', 'Grassland', 65, 320, True], ['S04', 'Ribbon', 'Kingsnake', ' Forest ', 95, 500, False]]

# Re-running this cell resets the raw CSV to the same starting data.
with CSV_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(COLUMNS)
    writer.writerows(ROWS)
print(f"Created {CSV_PATH}: {len(ROWS)} rows")

Created snake_data/snakes.csv: 13 rows


## 2. Load and inspect
A DataFrame is a labeled table; a Series is one labeled column. What is the difference between the row count and the number of unique snake IDs?

In [3]:
df = pd.read_csv(CSV_PATH)
print(df.head())
print("Shape (rows, columns):", df.shape)
print("Unique snake IDs:", df["snake_id"].nunique())
df.info()
print(df.describe())

  snake_id     name       species    habitat  length_cm  mass_g  venomous
0      S01   Noodle    Corn snake     Forest      110.0     650     False
1      S02  Pretzel   Ball python  Grassland      125.0    1400     False
2      S03    Socks  Garter snake    Wetland       55.0     140     False
3      S04   Ribbon     Kingsnake    Forest        95.0     500     False
4      S05    Mochi   Ball python  Grassland      100.0    1100     False
Shape (rows, columns): (13, 7)
Unique snake IDs: 12
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   snake_id   13 non-null     object 
 1   name       13 non-null     object 
 2   species    13 non-null     object 
 3   habitat    13 non-null     object 
 4   length_cm  12 non-null     float64
 5   mass_g     13 non-null     int64  
 6   venomous   13 non-null     bool   
dtypes: bool(1), float64(1), int64(1), obj

## 3. Grab columns and rows
Square brackets select columns. loc uses labels or conditions; iloc uses integer positions. With the default index, labels happen to look like positions, but they are not the same concept.

In [4]:
print(df["name"])                     # Series
print(df[["name", "length_cm"]])      # DataFrame
print(df.loc[0:2, ["name", "habitat"]]) # labels 0, 1, 2 (end included)
print(df.iloc[:3, :2])                 # first 3 rows, first 2 columns
print(df.loc[df["length_cm"] > 100, ["name", "length_cm"]])

0      Noodle
1     Pretzel
2       Socks
3      Ribbon
4       Mochi
5      Zigzag
6       Pesto
7      Pepper
8        Bean
9      Slinky
10     Pickle
11    Waffles
12     Ribbon
Name: name, dtype: object
       name  length_cm
0    Noodle      110.0
1   Pretzel      125.0
2     Socks       55.0
3    Ribbon       95.0
4     Mochi      100.0
5    Zigzag       70.0
6     Pesto       85.0
7    Pepper      150.0
8      Bean        NaN
9    Slinky       80.0
10   Pickle      120.0
11  Waffles       65.0
12   Ribbon       95.0
      name    habitat
0   Noodle     Forest
1  Pretzel  Grassland
2    Socks    Wetland
  snake_id     name
0      S01   Noodle
1      S02  Pretzel
2      S03    Socks
       name  length_cm
0    Noodle      110.0
1   Pretzel      125.0
7    Pepper      150.0
10   Pickle      120.0


## 4. Clean without inventing measurements
There is one repeated row, inconsistent habitat labels, and one missing length. Keep that unknown length; zero would be a made-up measurement. Later means skip missing lengths, so report how many measurements exist.

In [5]:
print("Missing values before cleaning:")
print(df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Raw habitats:", df["habitat"].unique())

clean = df.drop_duplicates().copy()
clean["habitat"] = clean["habitat"].str.strip().str.lower()
print("Clean shape:", clean.shape)
print("Clean habitat counts:")
print(clean["habitat"].value_counts())
print("Snakes missing a length:")
print(clean.loc[clean["length_cm"].isna(), ["name", "length_cm"]])

Missing values before cleaning:
snake_id     0
name         0
species      0
habitat      0
length_cm    1
mass_g       0
venomous     0
dtype: int64
Exact duplicate rows: 1
Raw habitats: ['Forest' 'Grassland' 'Wetland' ' Forest ' 'forest']
Clean shape: (12, 7)
Clean habitat counts:
habitat
forest       6
grassland    4
wetland      2
Name: count, dtype: int64
Snakes missing a length:
   name  length_cm
8  Bean        NaN


## 5. Filter and sort
Predict the names before running. A comparison creates a True/False mask. Use & for AND, | for OR, ~ for NOT, and parentheses around each comparison.

In [6]:
mask = (clean["length_cm"] > 100) & (~clean["venomous"])
long_nonvenomous = clean.loc[mask, ["name", "length_cm"]]
print(long_nonvenomous.sort_values("length_cm", ascending=False))
print("Top 3 recorded lengths:")
print(clean.sort_values("length_cm", ascending=False).head(3)[["name", "length_cm"]])

       name  length_cm
1   Pretzel      125.0
10   Pickle      120.0
0    Noodle      110.0
Top 3 recorded lengths:
       name  length_cm
7    Pepper      150.0
1   Pretzel      125.0
10   Pickle      120.0


## 6. Add, edit, and rename columns
Whole-column arithmetic applies to every row. Assign the result to keep a change. Use loc to update selected rows instead of chained indexing.

In [7]:
clean["length_m"] = clean["length_cm"] / 100
clean["size_band"] = "under 1 m"
clean.loc[clean["length_cm"] >= 100, "size_band"] = "1 m or longer"
clean.loc[clean["length_cm"].isna(), "size_band"] = "unknown"
clean = clean.rename(columns={"mass_g": "weight_g"})
print(clean[["name", "length_m", "weight_g", "size_band"]])

       name  length_m  weight_g      size_band
0    Noodle      1.10       650  1 m or longer
1   Pretzel      1.25      1400  1 m or longer
2     Socks      0.55       140      under 1 m
3    Ribbon      0.95       500      under 1 m
4     Mochi      1.00      1100  1 m or longer
5    Zigzag      0.70       400      under 1 m
6     Pesto      0.85       220      under 1 m
7    Pepper      1.50      1200  1 m or longer
8      Bean       NaN       450        unknown
9    Slinky      0.80       350      under 1 m
10   Pickle      1.20       750  1 m or longer
11  Waffles      0.65       320      under 1 m


## 7. Group, summarize, and export
Split into habitats, calculate within each group, combine the answers. size counts all snakes; count counts nonmissing lengths. These are descriptive summaries of made-up records, not species-level scientific conclusions.

In [8]:
summary = clean.groupby("habitat").agg(
    n_snakes=("snake_id", "size"),
    n_lengths=("length_cm", "count"),
    mean_length_cm=("length_cm", "mean"),
).round(1)
print(summary)
clean.to_csv(DATA_DIR / "snakes_clean.csv", index=False)
summary.to_csv(DATA_DIR / "habitat_summary.csv")  # keep habitat index
long_nonvenomous.to_csv(DATA_DIR / "long_nonvenomous.csv", index=False)
print("Saved clean data, habitat summary, and filtered results in", DATA_DIR)

           n_snakes  n_lengths  mean_length_cm
habitat                                       
forest            6          5            96.0
grassland         4          4           110.0
wetland           2          2            67.5
Saved clean data, habitat summary, and filtered results in snake_data


## 8. Your turn: snake detective
Work with a partner for 3 minutes. A: List forest snakes with recorded length >= 90 cm, longest first. B: Which habitat has the most snakes? C (stretch): Add weight_kg and find the heaviest nonvenomous snake. Write code below; solutions are in the facilitator guide.

In [9]:
# A: filter, select name + length_cm, then sort

# B: count snakes per habitat

# C (optional): derive weight_kg, filter, then sort


## Pocket reference
- `pd.read_csv(path)` / `df.to_csv(path, index=False)`: load/save a table.
- `df.head()`, `df.shape`, `df.info()`: inspect.
- `df["column"]`: one column; `df[["a", "b"]]`: several columns.
- `df.loc[condition, ["columns"]]`: filter rows and choose columns.
- `df.iloc[:3]`: first three rows by position.
- `df.isna().sum()`, `df.drop_duplicates()`: investigate missing and repeated records.
- `df.sort_values("column", ascending=False)`: largest first.
- `df.groupby("category")["number"].mean()`: mean by category.

Official documentation: [Getting started](https://pandas.pydata.org/docs/getting_started/intro_tutorials/), [selection](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html), [groupby](https://pandas.pydata.org/docs/user_guide/groupby.html).

Reminder: most transformations return a result; assign it to a variable to keep it.